# Deep Learning project - Group 8

This work was conducted by:

Alexandre Batista - 20250419 <br>
Daniel Caridade - 20211588 <br>
Luis Mendes - 20221949 <br>
Mehmet Karaca - 20250344 <br>
Veronica mendes - 20221945

# 0. Library installation & importation

__`Step 1`__ - Installing the required libraries. <br>

Note: Uncomment the cell below if the libraries are not already installed in your environment, otherwise leave it commented as it is.

In [1]:
# Library for creating a tree map
#!pip install squarify

# Library for performing perceptual hashing
#!pip install imagehash

__`Step 2`__ Loading the required libraries for the notebook.

In [4]:
import os
import sys
import tensorflow as tf
import matplotlib.pyplot as plt
import squarify
import hashlib
import imagehash
import numpy as np

from collections import defaultdict
from PIL import Image
from itertools import combinations
from pathlib import Path

# Add src folder to Python path so we can import project modules
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from utils import get_md5_hash, get_perceptual_hash

# 1. Data Integration

__`Step 3`__ - Setting the path to the local wikiart dataset.

In [ ]:
# Get the project root directory (one level up from notebooks folder)
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
wikiart_path = project_root / "data" / "wikiart"

print(f"Project root: {project_root}")
print(f"WikiArt path: {wikiart_path}")
print(f"WikiArt folder exists: {wikiart_path.exists()}")

__`Step 4`__ - Importing the dataset into Google Collab.

In [ ]:
# Create the dataset from the local directory
data_path = str(wikiart_path)

data = tf.keras.utils.image_dataset_from_directory(
    data_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=True,
    seed=42
)

# Print dataset the number and name of the classes
class_names = data.class_names
print(f"Classes: {class_names}")
print(f"Number of batches: {len(data)}")

Note for future me: All the images are 512x512, if the model is underperforming with 224x224 try using 512x512 instead. The drawbacks is that this will take more time to run, and maybe it will require me to do some adjustments especially when working with pre-trained models.

# 2. Data Understanding

__`Step 5`__ - Checking for class imbalance in the dataset.

In [ ]:
# Counting the number of images per class
class_counts = {}
for class_name in class_names:
    class_folder = os.path.join(data_path, class_name)
    class_counts[class_name] = len(os.listdir(class_folder))

total_samples = sum(class_counts.values())

# Printing the class distribution
print(f"Total samples: {total_samples}\n")
print("Class distribution:")
print("-" * 40)
for class_name, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_samples) * 100
    print(f"{class_name}: {count} samples ({percentage:.2f}%)")

# Treemap visualization
# Preparing the data for creating the treemap
labels = [f"{name}\n{count} ({count/total_samples*100:.1f}%)"
          for name, count in class_counts.items()]
sizes = list(class_counts.values())
colors = plt.cm.Spectral([i / len(class_counts) for i in range(len(class_counts))])

# Create the treemap
plt.figure(figsize=(10, 8))
squarify.plot(sizes=sizes, label=labels, color=colors, alpha=0.8,
              text_kwargs={'fontsize': 10, 'fontweight': 'bold'})
plt.title('Class Distribution (Percentage of Total Samples)', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

While there exists some class imbalanced, it is not very severe. The most represented class has fairly 10% of the total observations, while the least represented has 2.52% of the total observations.

This maybe a natural case of data imbalance, as different artists may have more works (paitings, drawings, portraits) than others, nonetheless oversampling or undersampling techniques might be beneficial if the performance in the least represented classes is significantly lower.

__`Step 6`__ - Checking for duplicates paths in the data.

In [ ]:
# Find all image hashes
hash_to_paths = defaultdict(list)

for class_name in class_names:
    class_folder = os.path.join(data_path, class_name)
    for img_name in os.listdir(class_folder):
        img_path = os.path.join(class_folder, img_name)
        try:
            img_hash = get_md5_hash(img_path)
            hash_to_paths[img_hash].append(img_path)
        except Exception as e:
            print(f"Error reading {img_path}: {e}")

# Find duplicates
duplicates = {h: paths for h, paths in hash_to_paths.items() if len(paths) > 1}

# Report results
total_images = sum(len(paths) for paths in hash_to_paths.values())
duplicate_count = sum(len(paths) - 1 for paths in duplicates.values())

print(f"Total images scanned: {total_images}")
print(f"Unique images: {total_images - duplicate_count}")
print(f"Duplicate images: {duplicate_count}")
print(f"Duplicate groups: {len(duplicates)}")

if duplicates:
    print("\nDuplicate groups found:")
    print("-" * 50)
    for i, (h, paths) in enumerate(duplicates.items(), 1):
        print(f"\nGroup {i} ({len(paths)} copies):")
        for p in paths:
            # Show relative path for readability
            rel_path = p.replace(data_path + '/', '')
            print(f"  {rel_path}")
else:
    print("\nNo duplicates found!")

__`Step 6.1`__ - Computing the parwise perceptual hash difference to better understand what can be the best threshold for the hash difference to including when comparing images that have similar visuals.

In [ ]:
all_diffs = []

for class_name in class_names:
    class_folder = os.path.join(data_path, class_name)
    image_hashes = []
    img_names = sorted(os.listdir(class_folder))
    for img_name in img_names:
        img_path = os.path.join(class_folder, img_name)
        img_hash = get_perceptual_hash(img_path)
        if img_hash is not None:
            image_hashes.append((img_path, img_hash))

    # Compare all pairs and store differences
    for i in range(len(image_hashes)):
        for j in range(i + 1, len(image_hashes)):
            diff = image_hashes[i][1] - image_hashes[j][1]
            all_diffs.append(diff)

print(f"Min diff: {min(all_diffs)}, Max diff: {max(all_diffs)}")

__`Step 6.1.1`__ - Plotting the histogram of the hash differences.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(all_diffs, bins=50)
plt.xlabel("Hash difference (Hamming distance)")
plt.ylabel("Number of image pairs")
plt.title("Distribution of perceptual hash differences")
plt.show()

After analysing the plot, their are relatively large hash differences between images. <br>
As the objective is to identify very similar images the following hash difference thresholds were tested: [20, 15, 10, 8, 7]. The code bellow only contains the hash difference value that was better suited for the task we were trying to do, the explanation for it is presented in the steps bellow.

__`Step 6.2`__ - Checking for very similar images in the dataset, trying to find masked duplicates.

In [ ]:
similar_pairs_per_class = {}
images_to_remove = []  # Store one image from each duplicate pair
hash_threshold = 7  # Images with hash difference <= 5 are considered similar

for class_name in class_names:
    print(f"Processing {class_name}...")
    class_folder = os.path.join(data_path, class_name)

    # Get all image paths and their hashes
    image_hashes = []
    img_names = sorted(os.listdir(class_folder))  # Sort filenames for determinism
    for img_name in img_names:
        img_path = os.path.join(class_folder, img_name)
        img_hash = get_perceptual_hash(img_path)
        if img_hash is not None:
            image_hashes.append((img_path, img_hash))

    # Compare all pairs within the class
    similar_pairs = []
    for i in range(len(image_hashes)):
        for j in range(i + 1, len(image_hashes)):
            path1, hash1 = image_hashes[i]
            path2, hash2 = image_hashes[j]
            diff = hash1 - hash2
            if diff <= hash_threshold and diff > 0:  # Similar but not identical
                # Sort paths to ensure consistency in storage
                path_pair = tuple(sorted([path1, path2]))
                similar_pairs.append((path_pair[0], path_pair[1], diff))
                images_to_remove.append(path_pair[1])  # Keep first, mark second for removal

    if similar_pairs:
        # Sort pairs by first path, then second path for consistent ordering
        similar_pairs_per_class[class_name] = sorted(similar_pairs, key=lambda x: (x[0], x[1]))

# Creating a variable that stores the path of the images to remove
images_to_remove = sorted(list(set(images_to_remove)))  # Sort for determinism

# Report results
print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)

total_pairs = sum(len(pairs) for pairs in similar_pairs_per_class.values())
print(f"\nTotal similar pairs found: {total_pairs}")
print(f"Unique images to remove: {len(images_to_remove)}")

for class_name in class_names:
    if class_name in similar_pairs_per_class:
        pairs = similar_pairs_per_class[class_name]
        print(f"\n{class_name}: {len(pairs)} similar pairs found")
    else:
        print(f"\n{class_name}: No similar images found")

# Visualize one example per class
classes_with_similar = [c for c in class_names if c in similar_pairs_per_class]

if classes_with_similar:
    n_classes = len(classes_with_similar)

    fig, axes = plt.subplots(n_classes, 2, figsize=(10, 5 * n_classes))

    if n_classes == 1:
        axes = [axes]

    for idx, class_name in enumerate(classes_with_similar):
        path1, path2, diff = similar_pairs_per_class[class_name][0]  # Always pick the first pair deterministically

        img1 = Image.open(path1)
        img2 = Image.open(path2)

        axes[idx][0].imshow(img1)
        axes[idx][0].set_title(f"{class_name}\n{os.path.basename(path1)}", fontsize=9)
        axes[idx][0].axis("off")

        axes[idx][1].imshow(img2)
        axes[idx][1].set_title(f"Hash diff: {diff}\n{os.path.basename(path2)}", fontsize=9)
        axes[idx][1].axis("off")

    fig.suptitle(
        "Similar Image Pairs (One Example Per Class)",
        fontsize=12,
        fontweight="bold",
        y=0.98
    )

    fig.subplots_adjust(hspace=0.4, wspace=0.3)
    fig.tight_layout(rect=[0, 0, 1, 0.98])

    plt.show()

else:
    print("\nNo similar images found in any class!")

While there aren't much, some very similar images where found. They are mainly the same painting with just a different shading of images. <br>

In total there are 87 duplicate pairs that need to be addressed.

__`Step 6.1.1`__ - Checking the pairs that were identified as similar to evaluate if there are false negatives or false positives.

In [ ]:
'''
# Flatten all similar pairs into a list with class info
all_pairs = []
for class_name, pairs in similar_pairs_per_class.items():
    for path1, path2, diff in pairs:
        all_pairs.append((class_name, path1, path2, diff))

if all_pairs:
    n_pairs = len(all_pairs)
    n_cols = 2  # Each pair has 2 images
    n_rows = n_pairs

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 5 * n_rows))

    # Normalize axes to always be 2D list: (n_rows, n_cols)
    if n_rows == 1:
        axes = [axes]
    axes = [ax_row if n_cols > 1 else [ax_row] for ax_row in axes]

    for idx, (class_name, path1, path2, diff) in enumerate(all_pairs):
        img1 = Image.open(path1)
        img2 = Image.open(path2)

        axes[idx][0].imshow(img1)
        axes[idx][0].set_title(f"{class_name}\n{os.path.basename(path1)}", fontsize=9)
        axes[idx][0].axis("off")

        axes[idx][1].imshow(img2)
        axes[idx][1].set_title(f"Hash diff: {diff}\n{os.path.basename(path2)}", fontsize=9)
        axes[idx][1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No similar pairs found.")
'''

From analysing the pairs of images and the after some experientation, using a hash difference of 7 gives us a almost perfect identification of duplicate images. All the pairs represent very short variations of the images with exception of 1 of the cases that represents a false positive in this method of identifying very similar (almost duplicated) images.

__`Step 7`__ - Checking how heterogeneous each class images are.

In [ ]:
# Calculate heterogeneity per class
heterogeneity_results = {}

for class_name in class_names:
    print(f"Processing {class_name}...")
    class_folder = os.path.join(data_path, class_name)

    if not os.path.isdir(class_folder):
        continue

    # Get all image hashes
    image_hashes = []
    for img_name in os.listdir(class_folder):
        img_path = os.path.join(class_folder, img_name)
        img_hash = get_perceptual_hash(img_path)
        if img_hash is not None:
            image_hashes.append((img_path, img_hash))

    if len(image_hashes) < 2:
        continue

    # Calculate all pairwise distances
    all_distances = []
    max_distance = 0
    most_different_pair = None

    for i in range(len(image_hashes)):
        for j in range(i + 1, len(image_hashes)):
            path1, hash1 = image_hashes[i]
            path2, hash2 = image_hashes[j]
            diff = hash1 - hash2
            all_distances.append(diff)

            if diff > max_distance:
                max_distance = diff
                most_different_pair = (path1, path2, diff)

    # Calculate heterogeneity metrics
    avg_distance = np.mean(all_distances)
    std_distance = np.std(all_distances)
    min_distance = np.min(all_distances)

    # Normalize heterogeneity score (0-100 scale, max possible hash diff is 64)
    heterogeneity_score = (avg_distance / 64) * 100

    heterogeneity_results[class_name] = {
        'score': heterogeneity_score,
        'avg_distance': avg_distance,
        'std_distance': std_distance,
        'min_distance': min_distance,
        'max_distance': max_distance,
        'most_different_pair': most_different_pair,
        'num_images': len(image_hashes)
    }

# Print results sorted by heterogeneity
print("\n" + "=" * 70)
print("HETEROGENEITY RESULTS (sorted by score)")
print("=" * 70)
print(f"{'Class':<25} {'Score':<10} {'Avg Dist':<10} {'Std':<10} {'Max Dist':<10}")
print("-" * 70)

sorted_results = sorted(heterogeneity_results.items(), key=lambda x: x[1]['score'], reverse=True)

for class_name, metrics in sorted_results:
    print(f"{class_name:<25} {metrics['score']:.2f}%     {metrics['avg_distance']:.2f}      {metrics['std_distance']:.2f}      {metrics['max_distance']}")

# Visualize most different pair per class
n_classes = len(heterogeneity_results)
fig, axes = plt.subplots(n_classes, 2, figsize=(10, 4 * n_classes))

if n_classes == 1:
    axes = [axes]

for idx, (class_name, metrics) in enumerate(sorted_results):
    path1, path2, diff = metrics['most_different_pair']

    img1 = Image.open(path1)
    img2 = Image.open(path2)

    axes[idx][0].imshow(img1)
    axes[idx][0].set_title(f"{class_name}\n{os.path.basename(path1)}", fontsize=9)
    axes[idx][0].axis('off')

    axes[idx][1].imshow(img2)
    axes[idx][1].set_title(f"Hash diff: {diff}\n{os.path.basename(path2)}", fontsize=9)
    axes[idx][1].axis('off')

fig.suptitle("Most Different Image Pairs Per Class", fontsize=14, fontweight='bold', y=0.98)
fig.subplots_adjust(hspace=0.4, wspace=0.2)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

From the code above we can conclude: <br>

1. All classes show very similar levels of internal variability, with heterogeneity scores ranging roughly between 47% and 49%. This indicates that the images within each artist’s class are visually diverse in terms of global composition and structure. For example, artists such as Edgar Degas, Salvador Dalí, and Claude Monet all present comparable average hash distances between their paintings. <br>
2. No class is significantly more homogeneous or heterogeneous than the others according to this metric. <br>
3. Overall, the dataset appears relatively balanced in terms of visual diversity within classes, meaning that the variability of artworks is consistent across different artists. However, since perceptual hashing mainly captures coarse structural similarities, these results reflect global image differences rather than deeper stylistic characteristics such as brushstroke patterns or artistic technique, so aditional inter categories analysis is required.

__`Step 7.1`__ - Checking inter cluster similarity.

In [ ]:
def get_perceptual_hash(image_path):
    try:
        img = Image.open(image_path)
        return imagehash.phash(img)
    except:
        return None

# Store hashes per class
class_hashes = {}

for class_name in class_names:
    print(f"Hashing images for {class_name}...")

    class_folder = os.path.join(data_path, class_name)

    if not os.path.isdir(class_folder):
        continue

    hashes = []

    for img_name in os.listdir(class_folder):
        img_path = os.path.join(class_folder, img_name)
        img_hash = get_perceptual_hash(img_path)

        if img_hash is not None:
            hashes.append(img_hash)

    if len(hashes) > 0:
        class_hashes[class_name] = hashes


# Compute inter-class distances
inter_class_results = {}

for classA, classB in combinations(class_hashes.keys(), 2):

    hashesA = class_hashes[classA]
    hashesB = class_hashes[classB]

    distances = []

    for h1 in hashesA:
        for h2 in hashesB:
            distances.append(h1 - h2)

    inter_class_results[(classA, classB)] = {
        "avg_distance": np.mean(distances),
        "std_distance": np.std(distances),
        "min_distance": np.min(distances),
        "max_distance": np.max(distances)
    }


# Sort by similarity (lowest distance = most similar classes)
sorted_results = sorted(
    inter_class_results.items(),
    key=lambda x: x[1]["avg_distance"]
)

print("\n" + "="*80)
print("INTER-CLASS SIMILARITY (Most Similar Artist Pairs)")
print("="*80)
print(f"{'Class A':<25} {'Class B':<25} {'Avg Dist':<10} {'Std':<10}")

for (classA, classB), metrics in sorted_results[:20]:
    print(f"{classA:<25} {classB:<25} {metrics['avg_distance']:.2f}      {metrics['std_distance']:.2f}")

From this code we can conclude (to edit):

1. Perceptual hash distances are very similar across and within classes. Both intra-class and inter-class average distances are around 31, meaning the metric does not strongly distinguish images belonging to the same artist from those belonging to different artists.
2. This does not necessarily indicate a problem for deep learning models.
Perceptual hashing captures only coarse global image structure, while deep learning models (e.g., CNNs) learn far richer features such as texture, brushstroke patterns, color distribution, and local spatial features.
3. Artists with similar artistic movements may appear visually similar at a global level.
For example, paintings by Claude Monet, Camille Pissarro, and Eugène Boudin can share similar landscape compositions, which explains the relatively small perceptual hash distances between them.
4. The dataset appears balanced in terms of visual variability.
Since no class shows extremely high or extremely low heterogeneity, the dataset does not appear biased toward overly uniform or overly diverse classes.
5. Potential difficulty may arise between stylistically related artists.
Deep learning models might occasionally confuse artists with similar themes or painting styles, but this is common in fine-grained classification tasks and can usually be mitigated with sufficient training data.


There is no strong evidence from the perceptual hash analysis that a deep learning model will fail at image classification. The results simply suggest that global visual similarity between paintings is relatively consistent across artists, meaning that the model will need to rely on more detailed stylistic features rather than basic structural differences.